# Module 06 — Convolutional Neural Networks

## 06-08 Neural Style Transfer

**Objective:** Implement optimization-based neural style transfer (Gatys et al.) from scratch — content loss on deep VGG features, style loss via Gram matrices, total-variation smoothing — then optimize raw pixels until one photograph is repainted with another image's textures, and dissect every knob (layer choice, style weight, initialization, color preservation) with controlled ablations.

**Prerequisites:** 06-02 (Convolution from Scratch), 06-03 (CNN Architectures — LeNet to ResNet), 06-04 (Transfer Learning & Fine-Tuning), 05-08 (Optimizers — SGD, Momentum & Nesterov)

## Part 0 — Setup & Prerequisites

This notebook covers neural style transfer: the discovery that a CNN trained only to classify objects implicitly factors images into *content* (what is depicted, encoded in deep feature maps) and *style* (textures, colors, brush statistics, encoded in feature **correlations**). We exploit that factorization by optimizing an image's pixels — not a network's weights — to match the content features of one image and the style statistics of another.

**Why it matters:** Style transfer is the classic demonstration that pretrained CNN features are a *perceptual representation*, not just classifier internals. The Gram-matrix idea reappears in texture synthesis, perceptual losses for super-resolution, and feature-statistics methods like AdaIN; and "optimize the input, freeze the network" is the same mechanism behind feature visualization and adversarial examples (06-10 will weaponize it).

Everything here is self-contained: content photos come from `sklearn.datasets.load_sample_image` and one style source is generated procedurally, so there are no external image downloads — only the standard pretrained VGG-19 weights from torchvision (as in 06-04).

In [ ]:
import sys
import math
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.datasets import load_sample_image

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"NumPy       : {np.__version__}")

In [ ]:
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Configuration. Optimization-based style transfer is compute-hungry: every step
# is a full VGG forward + backward. These sizes keep CPU runtimes civilized
# (roughly 10-20 min total for all runs); on GPU, feel free to double IMG_SIZE.
IMG_SIZE        = 192          # working resolution for the main runs
ABLATION_SIZE   = 128          # smaller resolution for sweeps/ablations
MAIN_STEPS      = 150          # Adam steps for the main runs
ABLATION_STEPS  = 80           # Adam steps for ablation runs
STYLE_WEIGHT    = 1e6          # beta: style loss multiplier (alpha = content = 1)
TV_WEIGHT       = 2e-5         # total-variation smoothing
LEARNING_RATE   = 0.05         # Adam on raw pixels

# VGG-19 was trained on ImageNet-normalized inputs; we must match it.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])

# Canonical Gatys layer choices on torchvision's vgg19.features indices:
#   conv1_1=0, conv2_1=5, conv3_1=10, conv4_1=19, conv4_2=21, conv5_1=28
STYLE_LAYERS   = [0, 5, 10, 19, 28]
CONTENT_LAYER  = 21
LAYER_NAMES    = {0: "conv1_1", 5: "conv2_1", 10: "conv3_1",
                  19: "conv4_1", 21: "conv4_2", 28: "conv5_1"}

**The images.** Content: scikit-learn's two bundled photographs (a temple facade and a flower). Styles: (1) the flower photo itself — photo-to-photo transfer is legitimate and shows color/texture statistics moving between real images — and (2) a procedurally generated "swirl" texture whose statistics are so distinctive that you can verify by eye exactly what the Gram matrices captured.

In [ ]:
def to_tensor(img_array: np.ndarray, size: int) -> torch.Tensor:
    """Convert an HWC uint8 image array to a resized, [0,1] float CHW tensor.

    Args:
        img_array: Image as a (H, W, 3) uint8 array.
        size: Target size for the shorter side; the image is center-cropped square.

    Returns:
        Tensor of shape (1, 3, size, size) in [0, 1].
    """
    t = torch.from_numpy(img_array.copy()).float().permute(2, 0, 1) / 255.0
    h, w = t.shape[1:]
    side = min(h, w)
    top, left = (h - side) // 2, (w - side) // 2
    t = t[:, top:top + side, left:left + side].unsqueeze(0)
    return F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)


def make_swirl_texture(size: int) -> torch.Tensor:
    """Generate a synthetic swirl-and-stripe texture as a style source.

    Args:
        size: Output image side length in pixels.

    Returns:
        Tensor of shape (1, 3, size, size) in [0, 1].
    """
    ys, xs = torch.meshgrid(
        torch.linspace(-3, 3, size), torch.linspace(-3, 3, size), indexing="ij"
    )
    radius = torch.sqrt(xs ** 2 + ys ** 2)
    angle = torch.atan2(ys, xs)
    swirl = torch.sin(6 * radius + 4 * angle)
    stripes = torch.sin(14 * xs + 5 * torch.sin(3 * ys))
    grain = 0.15 * torch.randn(size, size, generator=torch.Generator().manual_seed(SEED))
    r = 0.55 + 0.45 * swirl
    g = 0.35 + 0.35 * stripes
    b = 0.65 + 0.35 * torch.sin(8 * radius) + grain
    img = torch.stack([r, g, b]).clamp(0, 1).unsqueeze(0)
    return img


content_img = to_tensor(load_sample_image("china.jpg"), IMG_SIZE).to(device)
flower_img  = to_tensor(load_sample_image("flower.jpg"), IMG_SIZE).to(device)
swirl_img   = make_swirl_texture(IMG_SIZE).to(device)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, img, title in zip(
    axes, (content_img, flower_img, swirl_img),
    ("content: temple (china.jpg)", "style A: flower photo", "style B: procedural swirl"),
):
    ax.imshow(img[0].permute(1, 2, 0).cpu().numpy())
    ax.set_title(title, fontsize=9)
    ax.set_axis_off()
plt.tight_layout()
plt.show()
print(f"working tensors: {tuple(content_img.shape)} in [0,1] on {device}")

In [ ]:
# Frozen feature extractor: VGG-19 convolutional trunk, eval mode, no grads.
vgg = torchvision.models.vgg19(weights="IMAGENET1K_V1").features.to(device).eval()
for p in vgg.parameters():
    p.requires_grad_(False)

# Replace in-place ReLUs: we differentiate w.r.t. the INPUT image, and in-place
# ops would clobber activations needed by autograd.
for i, layer in enumerate(vgg):
    if isinstance(layer, nn.ReLU):
        vgg[i] = nn.ReLU(inplace=False)

n_params = sum(p.numel() for p in vgg.parameters())
print(f"VGG-19 features: {len(vgg)} layers, {n_params / 1e6:.1f}M frozen parameters")
print(f"style layers   : {[LAYER_NAMES[i] for i in STYLE_LAYERS]}")
print(f"content layer  : {LAYER_NAMES[CONTENT_LAYER]}")

**What the chosen layers actually see.** Before using these activations as loss targets, look at them. Early layers respond to edges and color patches; by `conv4_1`/`conv5_1` the maps are coarse, abstract, and object-ish — exactly why deep layers can hold "content" loosely enough for style to move in underneath. (We met these features as transfer-learning material in 06-04; here they become an optimization target.)

In [ ]:
def show_feature_maps(image: torch.Tensor, layers: list[int], n_channels: int) -> None:
    """Display the first few channels of VGG activations at several layers.

    Args:
        image: Input image, (1, 3, H, W) in [0, 1].
        layers: VGG layer indices to visualize (one row each).
        n_channels: Number of channels to show per layer.
    """
    feats = extract_features(normalize_for_vgg(image), layers)
    fig, axes = plt.subplots(len(layers), n_channels,
                             figsize=(1.55 * n_channels, 1.62 * len(layers)))
    for row, idx in enumerate(layers):
        fmap = feats[idx][0].detach().cpu()
        for col in range(n_channels):
            ax = axes[row, col]
            ax.imshow(fmap[col], cmap="viridis")
            ax.set_axis_off()
            if col == 0:
                ax.text(-0.32, 0.5, LAYER_NAMES.get(idx, str(idx)),
                        transform=ax.transAxes, rotation=90,
                        va="center", ha="center", fontsize=8)
        res = tuple(fmap.shape[-2:])
        axes[row, -1].set_title(f"{res[0]}x{res[1]}", fontsize=7, loc="right")
    fig.suptitle("Temple through VGG-19: edges -> textures -> abstract parts")
    plt.tight_layout()
    plt.show()


show_feature_maps(content_img, STYLE_LAYERS, n_channels=7)

## Part 1 — Content, Style, and the Gram Matrix from Scratch

Let $F^{\ell} \in \mathbb{R}^{C_\ell \times N_\ell}$ be layer $\ell$'s feature maps (channels × spatial positions, flattened) for some image.

**Content** is the features themselves. Two images share content at layer $\ell$ when their feature maps agree position by position:

$$\mathcal{L}_{\text{content}} = \tfrac{1}{2}\,\| F^{\ell}(x) - F^{\ell}(c) \|_2^2$$

Deep layers (we use `conv4_2`) care about object arrangement, not exact pixels — that slack is what lets style move in.

**Style** is *which features co-occur*, regardless of where. The Gram matrix erases position by correlating channels over all locations:

$$G^{\ell}_{ij} = \frac{1}{C_\ell H_\ell W_\ell} \sum_{p} F^{\ell}_{ip}\, F^{\ell}_{jp}
\qquad
\mathcal{L}_{\text{style}} = \sum_{\ell \in S} w_\ell\, \| G^{\ell}(x) - G^{\ell}(s) \|_2^2$$

"Channel 31 (vertical stroke) fires together with channel 14 (blue blob)" survives; "at the top-left corner" does not. Summing over early *and* late layers matches texture statistics at several scales at once.

We implement both losses plus total variation (a pixel-space smoother that suppresses optimization noise), and verify the vectorized Gram against an explicit loop.

In [ ]:
def extract_features(
    image: torch.Tensor,
    layers: list[int],
) -> dict[int, torch.Tensor]:
    """Run VGG once and collect feature maps at the requested layer indices.

    Args:
        image: Batch of shape (1, 3, H, W), ImageNet-normalized.
        layers: Indices into vgg.features whose outputs should be kept.

    Returns:
        Mapping from layer index to its (1, C, H', W') activation tensor.
    """
    feats: dict[int, torch.Tensor] = {}
    h = image
    last = max(layers)
    for i, layer in enumerate(vgg):
        h = layer(h)
        if i in layers:
            feats[i] = h
        if i == last:
            break
    return feats


def normalize_for_vgg(image: torch.Tensor) -> torch.Tensor:
    """Map a [0,1] RGB tensor to VGG's expected ImageNet statistics.

    Args:
        image: Batch of shape (1, 3, H, W) with values in [0, 1].

    Returns:
        Normalized tensor of the same shape.
    """
    mean = IMAGENET_MEAN.to(image.device).view(1, 3, 1, 1)
    std  = IMAGENET_STD.to(image.device).view(1, 3, 1, 1)
    return (image - mean) / std


def gram_matrix(features: torch.Tensor) -> torch.Tensor:
    """Channel-correlation Gram matrix, normalized by feature-map size.

    Args:
        features: Activation tensor of shape (1, C, H, W).

    Returns:
        Gram matrix of shape (C, C).
    """
    _, c, h, w = features.shape
    flat = features.view(c, h * w)
    return (flat @ flat.T) / (c * h * w)


def gram_matrix_loops(features: torch.Tensor) -> torch.Tensor:
    """Reference Gram computation with explicit loops (correctness oracle).

    Args:
        features: Activation tensor of shape (1, C, H, W).

    Returns:
        Gram matrix of shape (C, C), identical to gram_matrix up to float error.
    """
    _, c, h, w = features.shape
    flat = features.view(c, h * w)
    gram = torch.zeros(c, c, device=features.device)
    for i in range(c):
        for j in range(i, c):
            val = (flat[i] * flat[j]).sum()
            gram[i, j] = val
            gram[j, i] = val
    return gram / (c * h * w)


# Verify on a small real activation (conv1_1 of the content image, cropped).
small = extract_features(normalize_for_vgg(content_img), [0])[0][:, :16, :32, :32]
diff = (gram_matrix(small) - gram_matrix_loops(small)).abs().max().item()
print(f"max |vectorized - loop| Gram difference: {diff:.2e}")
assert diff < 1e-4, "vectorized Gram must match the explicit-loop reference"

In [ ]:
def content_loss(gen_feats: torch.Tensor, target_feats: torch.Tensor) -> torch.Tensor:
    """Mean squared error between generated and target content features.

    Args:
        gen_feats: Features of the image being optimized, (1, C, H, W).
        target_feats: Precomputed features of the content image, same shape.

    Returns:
        Scalar loss tensor.
    """
    return F.mse_loss(gen_feats, target_feats)


def style_loss(
    gen_feats: dict[int, torch.Tensor],
    target_grams: dict[int, torch.Tensor],
    layers: list[int],
) -> torch.Tensor:
    """Summed Gram-matrix MSE across the chosen style layers.

    Args:
        gen_feats: Layer-indexed features of the image being optimized.
        target_grams: Precomputed Gram matrices of the style image.
        layers: Style layer indices to include.

    Returns:
        Scalar loss tensor (equal layer weights).
    """
    loss = torch.zeros((), device=next(iter(gen_feats.values())).device)
    for i in layers:
        loss = loss + F.mse_loss(gram_matrix(gen_feats[i]), target_grams[i])
    return loss / len(layers)


def total_variation(image: torch.Tensor) -> torch.Tensor:
    """Anisotropic total variation: mean absolute neighbor difference.

    Args:
        image: Batch of shape (1, 3, H, W).

    Returns:
        Scalar smoothness penalty.
    """
    dh = (image[..., 1:, :] - image[..., :-1, :]).abs().mean()
    dw = (image[..., :, 1:] - image[..., :, :-1]).abs().mean()
    return dh + dw


# A first look at what the Gram matrix sees: style A vs style B at conv2_1.
feats_flower = extract_features(normalize_for_vgg(flower_img), STYLE_LAYERS)
feats_swirl  = extract_features(normalize_for_vgg(swirl_img),  STYLE_LAYERS)
g_flower = gram_matrix(feats_flower[5])[:48, :48].cpu()
g_swirl  = gram_matrix(feats_swirl[5])[:48, :48].cpu()

fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.8))
for ax, g, title in zip(axes, (g_flower, g_swirl),
                        ("flower photo — conv2_1 Gram", "swirl texture — conv2_1 Gram")):
    im = ax.imshow(g, cmap="magma")
    ax.set_title(title, fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()
print("different textures -> visibly different channel-correlation fingerprints")

**Gram distance is a style metric.** If Gram matrices really encode style, then *distances between Gram matrices* should sort images by texture, ignoring layout. Quick check: compare the temple photo against itself shifted (same style, different content alignment), the flower, and the swirl. The shifted temple should be nearest by far.

In [ ]:
def style_distance(img_a: torch.Tensor, img_b: torch.Tensor) -> float:
    """Aggregate Gram-matrix distance between two images across style layers.

    Args:
        img_a: First image, (1, 3, H, W) in [0, 1].
        img_b: Second image, same shape.

    Returns:
        Mean per-layer Frobenius distance between Gram matrices.
    """
    fa = extract_features(normalize_for_vgg(img_a), STYLE_LAYERS)
    fb = extract_features(normalize_for_vgg(img_b), STYLE_LAYERS)
    dists = [
        (gram_matrix(fa[i]) - gram_matrix(fb[i])).norm().item()
        for i in STYLE_LAYERS
    ]
    return float(np.mean(dists))


shifted_temple = torch.roll(content_img, shifts=40, dims=3)
candidates = {
    "temple, shifted 40px": shifted_temple,
    "flower photo":         flower_img,
    "swirl texture":        swirl_img,
}
rows = [{"image": name, "gram distance to temple": style_distance(content_img, img)}
        for name, img in candidates.items()]
dist_df = pd.DataFrame(rows).sort_values("gram distance to temple")
print(dist_df.to_string(index=False, formatters={
    "gram distance to temple": "{:.4f}".format,
}))
print("\nposition moved, style metric barely flinched — Gram matrices ignore layout")

## Part 2 — Putting It All Together

The optimizer below is the whole algorithm: freeze VGG, precompute the content features and style Grams once, then repeatedly (1) extract features of the current canvas, (2) combine $\mathcal{L} = \alpha\,\mathcal{L}_{\text{content}} + \beta\,\mathcal{L}_{\text{style}} + \gamma\,\mathcal{L}_{\text{TV}}$, (3) backprop **into the pixels**, (4) Adam-step the canvas and clamp to $[0,1]$. We sanity-check with a 10-step micro-run before spending real compute.

In [ ]:
def make_canvas(content: torch.Tensor, init: str | torch.Tensor) -> torch.Tensor:
    """Build the starting canvas for pixel optimization.

    Args:
        content: Content image, (1, 3, H, W) in [0, 1] — defines shape/device.
        init: 'content' to start from the content image, 'noise' for random
            pixels, or an explicit tensor (e.g. an upsampled coarse result).

    Returns:
        A leaf tensor with requires_grad enabled, same shape as `content`.
    """
    if isinstance(init, torch.Tensor):
        canvas = F.interpolate(init.to(content.device), size=content.shape[-2:],
                               mode="bilinear", align_corners=False).clone()
    elif init == "content":
        canvas = content.clone()
    else:
        gen_cpu = torch.Generator().manual_seed(SEED)
        canvas = torch.rand(content.shape, generator=gen_cpu).to(content.device)
    return canvas.requires_grad_(True)


def run_style_transfer(
    content: torch.Tensor,
    style: torch.Tensor,
    steps: int,
    style_weight: float = STYLE_WEIGHT,
    content_weight: float = 1.0,
    tv_weight: float = TV_WEIGHT,
    content_layer: int = CONTENT_LAYER,
    style_layers: list[int] | None = None,
    init: str | torch.Tensor = "content",
    lr: float = LEARNING_RATE,
    log_every: int = 0,
) -> tuple[torch.Tensor, pd.DataFrame]:
    """Optimize an image to match `content`'s features and `style`'s Grams.

    Args:
        content: Content image, (1, 3, H, W) in [0, 1].
        style: Style image, same shape conventions.
        steps: Number of Adam steps on the pixels.
        style_weight: Multiplier beta on the style loss.
        content_weight: Multiplier alpha on the content loss (0 = pure texture
            synthesis).
        tv_weight: Multiplier on the total-variation smoother.
        content_layer: VGG layer index for the content loss.
        style_layers: VGG layer indices for the style loss (default canon set).
        init: 'content', 'noise', or an explicit starting canvas tensor.
        lr: Adam learning rate on the canvas.
        log_every: If > 0, print losses every this many steps.

    Returns:
        Tuple of (final image tensor detached to CPU, per-step loss DataFrame
        including a per-layer style-loss column for each style layer).
    """
    layers = STYLE_LAYERS if style_layers is None else style_layers
    needed = sorted(set(layers + [content_layer]))

    with torch.no_grad():
        target_content = extract_features(
            normalize_for_vgg(content), [content_layer]
        )[content_layer]
        style_feats = extract_features(normalize_for_vgg(style), layers)
        target_grams = {i: gram_matrix(style_feats[i]) for i in layers}

    canvas = make_canvas(content, init)
    optimizer = torch.optim.Adam([canvas], lr=lr)
    history: list[dict[str, float]] = []
    for step in range(1, steps + 1):
        feats = extract_features(normalize_for_vgg(canvas), needed)
        l_content = content_loss(feats[content_layer], target_content)
        per_layer = {
            i: F.mse_loss(gram_matrix(feats[i]), target_grams[i]) for i in layers
        }
        l_style = sum(per_layer.values()) / len(layers)
        l_tv = total_variation(canvas)
        loss = content_weight * l_content + style_weight * l_style + tv_weight * l_tv

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            canvas.clamp_(0, 1)

        record = {
            "step": step,
            "content": l_content.item(),
            "style (x beta)": style_weight * l_style.item(),
            "total": loss.item(),
        }
        for i, val in per_layer.items():
            record[f"style@{LAYER_NAMES.get(i, i)}"] = val.item()
        history.append(record)
        if log_every and step % log_every == 0:
            print(f"  step {step:>4}/{steps}  content {l_content.item():.4f}  "
                  f"style*beta {style_weight * l_style.item():.4f}")
    return canvas.detach().cpu(), pd.DataFrame(history)


def show_images(images: list[torch.Tensor], titles: list[str], suptitle: str) -> None:
    """Display a row of image tensors side by side.

    Args:
        images: List of (1, 3, H, W) tensors in [0, 1].
        titles: One title per image.
        suptitle: Figure-level caption.
    """
    fig, axes = plt.subplots(1, len(images), figsize=(3.6 * len(images), 3.8))
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1).numpy())
        ax.set_title(title, fontsize=9)
        ax.set_axis_off()
    fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


# Micro sanity run: 10 steps must reduce the total loss.
t0 = time.time()
_, micro_hist = run_style_transfer(content_img, swirl_img, steps=10)
print(f"10 steps in {time.time() - t0:.1f}s "
      f"({(time.time() - t0) / 10:.2f}s/step at {IMG_SIZE}px on {device})")
print(f"total loss step 1 -> 10: {micro_hist['total'].iloc[0]:.3f} -> "
      f"{micro_hist['total'].iloc[-1]:.3f}")
assert micro_hist["total"].iloc[-1] < micro_hist["total"].iloc[0], \
    "optimization must reduce the combined loss"

## Part 3 — Repainting the Temple

Two full runs at working resolution: the temple repainted with the flower photo's statistics, and with the swirl texture. Watch the loss decomposition — style loss collapses by orders of magnitude in the first third of optimization, while content loss *rises slightly from zero* (we start at the content image, then trade exact content for style).

In [ ]:
print("run 1/2: temple <- flower-photo style")
stylized_flower, hist_flower = run_style_transfer(
    content_img, flower_img, steps=MAIN_STEPS, log_every=50
)
print("run 2/2: temple <- swirl-texture style")
stylized_swirl, hist_swirl = run_style_transfer(
    content_img, swirl_img, steps=MAIN_STEPS, log_every=50
)

show_images(
    [content_img.cpu(), stylized_flower, stylized_swirl],
    ["original", "flower-photo style", "swirl-texture style"],
    "Optimization-based style transfer (Gatys), %d steps" % MAIN_STEPS,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
for ax, hist, title in zip(axes, (hist_flower, hist_swirl),
                           ("temple <- flower", "temple <- swirl")):
    ax.semilogy(hist["step"], hist["content"], label="content")
    ax.semilogy(hist["step"], hist["style (x beta)"], label="style x beta")
    ax.semilogy(hist["step"], hist["total"], label="total", ls="--", color="black")
    ax.set_xlabel("step"); ax.set_ylabel("loss (log)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print("style term dominates early and collapses fast; content rises from zero as")
print("the canvas leaves the content image, then both settle into a truce.")

# Which scale converges first? Per-layer style losses from the swirl run.
layer_cols = [c for c in hist_swirl.columns if c.startswith("style@")]
fig, ax = plt.subplots(figsize=(6.6, 3.6))
for col in layer_cols:
    ax.semilogy(hist_swirl["step"], hist_swirl[col], label=col.replace("style@", ""))
ax.set_xlabel("step"); ax.set_ylabel("per-layer style loss (log)")
ax.set_title("Coarse layers settle quickly; fine-texture layers keep refining")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

**Texture synthesis: style transfer's ancestor.** Set the content weight to zero, start from noise, and the same machinery becomes Gatys' 2015 texture synthesizer — it dreams a brand-new image with the style's statistics and *nobody's* content. This isolates what the Gram targets contain on their own, and proves the content term is what kept the temple visible above.

In [ ]:
print("pure texture synthesis from noise (content_weight = 0)")
texture_only, hist_tex = run_style_transfer(
    content_img, swirl_img, steps=MAIN_STEPS,
    content_weight=0.0, init="noise", tv_weight=1e-5,
)
show_images(
    [swirl_img.cpu(), texture_only],
    ["style source", "synthesized texture (no content)"],
    "Gram statistics alone generate texture — content is a separate, optional term",
)
print(f"final style loss: {hist_tex['style (x beta)'].iloc[-1]:.4f} "
      f"(vs {hist_swirl['style (x beta)'].iloc[-1]:.4f} when content competes)")

**Two practical upgrades, measured.**

*Coarse-to-fine:* optimize at half resolution first, upsample, and finish at full resolution — large motifs form cheaply at low res, fine detail is added late. *L-BFGS:* the second-order optimizer used in the original paper and most reference implementations; it typically reaches a lower loss in fewer steps than Adam, at a higher cost per step. We run both against our Adam baseline with matched wall-clock awareness.

In [ ]:
# (a) Coarse-to-fine: 96px for the first stage, finish at IMG_SIZE.
content_coarse = F.interpolate(content_img, size=(96, 96),
                               mode="bilinear", align_corners=False)
swirl_coarse = F.interpolate(swirl_img, size=(96, 96),
                             mode="bilinear", align_corners=False)
print("stage 1/2: 96px")
coarse_result, _ = run_style_transfer(content_coarse, swirl_coarse,
                                      steps=ABLATION_STEPS)
print("stage 2/2: refine at full working resolution")
c2f_result, hist_c2f = run_style_transfer(
    content_img, swirl_img, steps=MAIN_STEPS // 2, init=coarse_result
)
show_images(
    [stylized_swirl, c2f_result],
    [f"single-scale ({MAIN_STEPS} steps)",
     f"coarse-to-fine ({ABLATION_STEPS}+{MAIN_STEPS // 2} steps)"],
    "Coarse-to-fine reaches comparable quality with cheaper high-res steps",
)
print(f"final total loss — single-scale {hist_swirl['total'].iloc[-1]:.3f}  "
      f"vs coarse-to-fine {hist_c2f['total'].iloc[-1]:.3f}")

In [ ]:
def run_style_transfer_lbfgs(
    content: torch.Tensor,
    style: torch.Tensor,
    steps: int,
    style_weight: float = STYLE_WEIGHT,
    tv_weight: float = TV_WEIGHT,
) -> tuple[torch.Tensor, list[float]]:
    """Style transfer optimized with L-BFGS closures (reference-style setup).

    Args:
        content: Content image, (1, 3, H, W) in [0, 1].
        style: Style image, same conventions.
        steps: Number of optimizer.step() calls (each may evaluate several times).
        style_weight: Style loss multiplier.
        tv_weight: Total-variation multiplier.

    Returns:
        Tuple of (final image on CPU, recorded total loss per step).
    """
    with torch.no_grad():
        target_content = extract_features(
            normalize_for_vgg(content), [CONTENT_LAYER]
        )[CONTENT_LAYER]
        style_feats = extract_features(normalize_for_vgg(style), STYLE_LAYERS)
        target_grams = {i: gram_matrix(style_feats[i]) for i in STYLE_LAYERS}

    canvas = content.clone().requires_grad_(True)
    optimizer = torch.optim.LBFGS([canvas], max_iter=4, history_size=10)
    losses: list[float] = []

    for _ in range(steps):
        def closure() -> torch.Tensor:
            """Recompute the combined loss and its pixel gradients."""
            optimizer.zero_grad(set_to_none=True)
            feats = extract_features(normalize_for_vgg(canvas),
                                     sorted(set(STYLE_LAYERS + [CONTENT_LAYER])))
            loss = (
                content_loss(feats[CONTENT_LAYER], target_content)
                + style_weight * style_loss(feats, target_grams, STYLE_LAYERS)
                + tv_weight * total_variation(canvas)
            )
            loss.backward()
            return loss

        loss_val = optimizer.step(closure)
        losses.append(float(loss_val))
        with torch.no_grad():
            canvas.clamp_(0, 1)
    return canvas.detach().cpu(), losses


# Ablation-resolution copies, used here and throughout Part 4.
content_small = F.interpolate(content_img, size=(ABLATION_SIZE, ABLATION_SIZE),
                              mode="bilinear", align_corners=False)
swirl_small = F.interpolate(swirl_img, size=(ABLATION_SIZE, ABLATION_SIZE),
                            mode="bilinear", align_corners=False)

t0 = time.time()
lbfgs_img, lbfgs_losses = run_style_transfer_lbfgs(
    content_small, swirl_small, steps=ABLATION_STEPS // 4
)
lbfgs_time = time.time() - t0

t0 = time.time()
adam_img, adam_hist = run_style_transfer(content_small, swirl_small,
                                         steps=ABLATION_STEPS)
adam_time = time.time() - t0

print(f"L-BFGS: {lbfgs_losses[-1]:.3f} final loss in {lbfgs_time:.0f}s "
      f"({ABLATION_STEPS // 4} steps x ~4 evals)")
print(f"Adam  : {adam_hist['total'].iloc[-1]:.3f} final loss in {adam_time:.0f}s "
      f"({ABLATION_STEPS} steps)")
show_images([adam_img, lbfgs_img], ["Adam", "L-BFGS"],
            "Same losses, two optimizers — L-BFGS earns its reputation on this problem")

## Part 4 — Evaluation & Analysis

Style transfer has no accuracy number; evaluation means **controlled ablations** plus the quantitative loss/style-distance metrics we built. Five questions:

1. What does the content/style trade-off $\beta$ actually control? (sweep)
2. Which layers carry style at which scale? (early-only vs late-only vs all)
3. Does initialization matter? (content-init vs noise-init)
4. Did the output really acquire the style, *measurably*? (Gram distances before/after)
5. Can styles be blended? (Gram-target interpolation)
6. Does the style image's own scale matter? (feeding the style at 64px vs 256px)
7. Can we keep the original colors while transferring texture? (luminance-only transfer + histograms)

In [ ]:
# 1) Style-weight sweep at ablation resolution (content_small/swirl_small
# were prepared for the optimizer comparison above).
sweep_imgs, sweep_titles = [], []
for beta in (1e4, 1e6, 1e8):
    print(f"sweep: beta = {beta:.0e}")
    img, _ = run_style_transfer(content_small, swirl_small,
                                steps=ABLATION_STEPS, style_weight=beta)
    sweep_imgs.append(img)
    sweep_titles.append(f"beta = {beta:.0e}")
show_images(sweep_imgs, sweep_titles,
            "Style weight: photograph -> compromise -> pure texture")

**Reading the sweep.** At $\beta = 10^4$ the content term wins — the temple barely changes. At $10^8$ the style term wins — the architecture dissolves into texture. The interesting outputs live in between, which is why $\beta$ (or the ratio $\alpha/\beta$) is *the* knob users actually turn in style-transfer tools.

In [ ]:
# 2) Layer ablation: style from early layers only, late only, and all.
layer_sets = {
    "early only (conv1_1, conv2_1)": [0, 5],
    "late only (conv4_1, conv5_1)":  [19, 28],
    "all five (canon)":              STYLE_LAYERS,
}
layer_imgs, layer_titles = [], []
for name, layer_set in layer_sets.items():
    print(f"layer ablation: {name}")
    img, _ = run_style_transfer(content_small, swirl_small,
                                steps=ABLATION_STEPS, style_layers=layer_set)
    layer_imgs.append(img)
    layer_titles.append(name)
show_images(layer_imgs, layer_titles,
            "Style layers select texture scale: fine grain vs large motifs vs both")

In [ ]:
# 3) Initialization: same target, two starting canvases.
print("init ablation: content start")
img_content_init, hist_ci = run_style_transfer(
    content_small, swirl_small, steps=ABLATION_STEPS, init="content"
)
print("init ablation: noise start")
img_noise_init, hist_ni = run_style_transfer(
    content_small, swirl_small, steps=ABLATION_STEPS, init="noise"
)
show_images(
    [img_content_init, img_noise_init],
    ["init = content image", "init = random noise"],
    "Same losses, different basins: noise-init keeps less content structure",
)
print(f"final total loss — content-init {hist_ci['total'].iloc[-1]:.3f}  "
      f"vs noise-init {hist_ni['total'].iloc[-1]:.3f}")

**4) The numbers behind the pictures.** The stylized temple should have moved *toward the style image* in Gram space while staying *close to the content image* in deep-feature space. Both motions are measurable with the metrics from Part 1 — this is the honest scorecard of a successful transfer.

In [ ]:
def content_distance(img_a: torch.Tensor, img_b: torch.Tensor) -> float:
    """Distance between deep content features of two images.

    Args:
        img_a: First image, (1, 3, H, W) in [0, 1].
        img_b: Second image, same shape.

    Returns:
        MSE between conv4_2 feature maps.
    """
    fa = extract_features(normalize_for_vgg(img_a), [CONTENT_LAYER])[CONTENT_LAYER]
    fb = extract_features(normalize_for_vgg(img_b), [CONTENT_LAYER])[CONTENT_LAYER]
    return F.mse_loss(fa, fb).item()


stylized_dev = stylized_swirl.to(device)
scorecard = pd.DataFrame([
    {"pair": "temple vs swirl (before)",
     "gram dist": style_distance(content_img, swirl_img),
     "content dist": content_distance(content_img, swirl_img)},
    {"pair": "stylized vs swirl (after)",
     "gram dist": style_distance(stylized_dev, swirl_img),
     "content dist": content_distance(stylized_dev, swirl_img)},
    {"pair": "stylized vs temple",
     "gram dist": style_distance(stylized_dev, content_img),
     "content dist": content_distance(stylized_dev, content_img)},
])
print(scorecard.to_string(index=False, formatters={
    "gram dist": "{:.4f}".format, "content dist": "{:.4f}".format,
}))
print("\nstylized image: an order closer to the style in Gram space than the")
print("original was, while staying far closer to the temple in content space.")

**5) Style interpolation.** Gram targets live in a vector space, so we can *blend* two styles: optimize against $\lambda\, G^{\ell}(s_1) + (1 - \lambda)\, G^{\ell}(s_2)$. The midpoint isn't a double exposure — it's a coherent in-between texture, which is good evidence that Gram space is a meaningful style representation rather than a lookup table.

In [ ]:
def run_interpolated_style(
    content: torch.Tensor,
    style_a: torch.Tensor,
    style_b: torch.Tensor,
    lam: float,
    steps: int,
) -> torch.Tensor:
    """Style transfer against a convex combination of two styles' Gram targets.

    Args:
        content: Content image, (1, 3, H, W) in [0, 1].
        style_a: First style image (weight lam).
        style_b: Second style image (weight 1 - lam).
        lam: Blend factor in [0, 1].
        steps: Adam steps on the pixels.

    Returns:
        Final stylized image on CPU.
    """
    with torch.no_grad():
        target_content = extract_features(
            normalize_for_vgg(content), [CONTENT_LAYER]
        )[CONTENT_LAYER]
        feats_a = extract_features(normalize_for_vgg(style_a), STYLE_LAYERS)
        feats_b = extract_features(normalize_for_vgg(style_b), STYLE_LAYERS)
        target_grams = {
            i: lam * gram_matrix(feats_a[i]) + (1 - lam) * gram_matrix(feats_b[i])
            for i in STYLE_LAYERS
        }

    canvas = content.clone().requires_grad_(True)
    optimizer = torch.optim.Adam([canvas], lr=LEARNING_RATE)
    for _ in range(steps):
        feats = extract_features(normalize_for_vgg(canvas),
                                 sorted(set(STYLE_LAYERS + [CONTENT_LAYER])))
        loss = (
            content_loss(feats[CONTENT_LAYER], target_content)
            + STYLE_WEIGHT * style_loss(feats, target_grams, STYLE_LAYERS)
            + TV_WEIGHT * total_variation(canvas)
        )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            canvas.clamp_(0, 1)
    return canvas.detach().cpu()


flower_small = F.interpolate(flower_img, size=(ABLATION_SIZE, ABLATION_SIZE),
                             mode="bilinear", align_corners=False)
blend_imgs, blend_titles = [], []
for lam in (1.0, 0.5, 0.0):
    print(f"interpolation: lambda = {lam}")
    blend_imgs.append(run_interpolated_style(
        content_small, swirl_small, flower_small, lam, ABLATION_STEPS
    ))
    blend_titles.append(f"lambda = {lam} " +
                        ("(pure swirl)" if lam == 1 else
                         "(pure flower)" if lam == 0 else "(blend)"))
show_images(blend_imgs, blend_titles,
            "Interpolating Gram targets blends styles coherently")

**6) Style scale matters.** The Gram statistics are computed at whatever resolution we feed the style image. Feeding the *same* style at 64px vs 256px changes the physical size of its motifs relative to the canvas — a frequent gotcha when users plug in arbitrary style photos.

In [ ]:
scale_imgs, scale_titles = [], []
for style_res in (64, 256):
    style_scaled = F.interpolate(swirl_img, size=(style_res, style_res),
                                 mode="bilinear", align_corners=False)
    print(f"style fed at {style_res}px (canvas stays {ABLATION_SIZE}px)")
    img, _ = run_style_transfer(content_small, style_scaled,
                                steps=ABLATION_STEPS)
    scale_imgs.append(img)
    scale_titles.append(f"style @ {style_res}px")
show_images(scale_imgs, scale_titles,
            "Same style image, different input scale -> different motif size")

**7) Color-preserving transfer.** Classic trick (Gatys et al. 2016b): transfer style only in *luminance*. Convert both images to YCbCr-style channels, run the transfer logic's output luminance against the content's chrominance — texture moves, the temple keeps its own palette. We implement the simplest variant: recombine the stylized luminance with the content's color channels.

In [ ]:
def rgb_to_ycbcr(image: torch.Tensor) -> torch.Tensor:
    """Convert an RGB tensor in [0,1] to YCbCr (BT.601 coefficients).

    Args:
        image: Batch of shape (1, 3, H, W).

    Returns:
        Tensor of the same shape with channels (Y, Cb, Cr).
    """
    r, g, b = image[:, 0], image[:, 1], image[:, 2]
    y  = 0.299 * r + 0.587 * g + 0.114 * b
    cb = 0.5 - 0.168736 * r - 0.331264 * g + 0.5 * b
    cr = 0.5 + 0.5 * r - 0.418688 * g - 0.081312 * b
    return torch.stack([y, cb, cr], dim=1)


def ycbcr_to_rgb(image: torch.Tensor) -> torch.Tensor:
    """Convert a YCbCr tensor back to RGB in [0,1].

    Args:
        image: Batch of shape (1, 3, H, W) with channels (Y, Cb, Cr).

    Returns:
        RGB tensor of the same shape, clamped to [0, 1].
    """
    y, cb, cr = image[:, 0], image[:, 1] - 0.5, image[:, 2] - 0.5
    r = y + 1.402 * cr
    g = y - 0.344136 * cb - 0.714136 * cr
    b = y + 1.772 * cb
    return torch.stack([r, g, b], dim=1).clamp(0, 1)


stylized_y = rgb_to_ycbcr(stylized_swirl)[:, 0:1]
content_cbcr = rgb_to_ycbcr(content_img.cpu())[:, 1:]
color_preserved = ycbcr_to_rgb(torch.cat([stylized_y, content_cbcr], dim=1))

show_images(
    [content_img.cpu(), stylized_swirl, color_preserved],
    ["original", "full transfer", "luminance-only transfer"],
    "Color preservation: texture moves, the temple keeps its palette",
)

# Channel histograms quantify the palette shift and its repair.
def channel_histograms(image: torch.Tensor, bins: int = 40) -> np.ndarray:
    """Per-channel intensity histograms of an RGB image.

    Args:
        image: Batch of shape (1, 3, H, W) in [0, 1].
        bins: Number of histogram bins.

    Returns:
        Array of shape (3, bins) with normalized counts.
    """
    hists = []
    for ch in range(3):
        h, _ = np.histogram(image[0, ch].numpy().ravel(), bins=bins, range=(0, 1))
        hists.append(h / h.sum())
    return np.stack(hists)


named = {
    "original": content_img.cpu(),
    "full transfer": stylized_swirl,
    "luminance-only": color_preserved,
}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.0), sharey=True)
for ax, (name, img) in zip(axes, named.items()):
    hists = channel_histograms(img)
    centers = np.linspace(0, 1, hists.shape[1])
    for ch, color in enumerate(("red", "green", "blue")):
        ax.plot(centers, hists[ch], color=color, lw=1.2)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("intensity")
axes[0].set_ylabel("frequency")
fig.suptitle("Full transfer rewrites the color distribution; luminance-only restores it")
plt.tight_layout()
plt.show()

**Error analysis — when style transfer fails.** Three standing failure modes, all visible in our runs if you look closely: (1) **content bleed-through** at high $\beta$ — strong edges of the temple ghost through pure texture because deep content features resist; (2) **texture seams/blotches** when TV weight is too low (rerun a sweep cell with `tv_weight=0` and the sky fills with high-frequency confetti); (3) **scale mismatch** — the swirl's motifs were generated at our working resolution, but using a style image much larger/smaller than the canvas transfers textures at the wrong scale (the layer ablation showed which layers would mediate that). The fixes practitioners use — TV/feature regularizers, multi-resolution optimization, feed-forward networks trained with these same losses (Johnson et al.) — all start from the loss design we built here.

## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **A classifier's features double as a perceptual code.** Deep VGG activations encode *what and where* (content); their channel correlations — Gram matrices — encode *which textures, regardless of where* (style). Style transfer is just gradient descent on pixels under both constraints at once.
- **The Gram matrix discards position by construction**: shifting the temple 40px barely moved our Gram-distance metric, while a different texture moved it an order of magnitude. That position-blindness is exactly why it isolates style.
- **$\beta$ trades photograph for texture**, and the useful range spans orders of magnitude ($10^4$–$10^8$ here) because the two losses live on different scales.
- **Layer choice selects texture scale**: early-layer style transfers fine grain, late-layer style transfers large motifs; the canon five-layer set matches statistics across scales simultaneously.
- **Optimize-the-input is a general mechanism.** Freeze the network, differentiate w.r.t. pixels, and you get style transfer with this loss, feature visualization with another — and adversarial examples with a malicious one.

### What's Next

→ **06-09 (1D & 3D Convolutions)** leaves images briefly to generalize the convolution machinery across dimensions; then **06-10 (Adversarial Examples & Robustness)** reuses this notebook's optimize-the-input mechanism with hostile intent — the same gradients that painted the temple will break the classifier.

### Going Further

- Gatys, Ecker & Bethge, *A Neural Algorithm of Artistic Style* (2015) — the original formulation reproduced here.
- Johnson, Alahi & Fei-Fei, *Perceptual Losses for Real-Time Style Transfer* (2016) — train a feed-forward net with these losses; 1000× faster inference.
- Huang & Belongie, *AdaIN* (2017) — style as feature mean/variance, arbitrary styles in one pass.
- Gatys et al., *Preserving Color in Neural Artistic Style Transfer* (2016) — the luminance trick from Part 4, plus color-histogram matching.